# Theorem 2 synthetic selected-publication experiment

This notebook runs a resumable synthetic theorem-2 experiment with:
- tokens `1, ..., M`
- all 1-grams available
- random 2-grams kept with probability `1/2`
- random 3-grams kept with probability `1/4`
- desirable and undesirable `r`-grams
- a publication filter that always keeps desirable trajectories and vetoes undesirable ones

The run is saved under a versioned output directory with:
- checkpoint state for restart
- partial and final metrics CSV files
- saved figures with version and timestamp in the title
- the sampled pattern library and a sample-sequence text report


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, Markdown, display


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if p.name == "Drift_and_selection":
            return p
        if (p / "GitHub").exists() and (p / "Nat_Paper").exists():
            return p
    return start


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "GitHub" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from drift_selection.theorem2_selected_publication import (
    ExperimentConfig,
    PatternLibrary,
    PublicationPolicy,
    SyntheticGrammarConfig,
    default_output_root,
    format_pattern,
    run_publication_experiment,
)

pd.set_option("display.max_columns", 100)


## Configuration

This cell keeps the user-specified setup simple:
- `M = 100`
- `p(2-gram) = 1/2`
- `p(3-gram) = 1/4`
- `r = 5`
- selected publication keeps desirable and neutral trajectories, and vetoes undesirable or mixed ones

The remaining controls are operational rather than conceptual: number of rounds, candidates per round, evaluation size, and the experiment version tag.


In [ ]:
EXPERIMENT_VERSION = "V0_01"
SEED = 123

M = 100
R_SPAN = 5
DESIRABLE_COUNT = 4
UNDESIRABLE_COUNT = 4

ROUNDS = 6
CANDIDATES_PER_ROUND = 300
EVALUATION_COUNT = 160
SEARCH_TRIALS = 16

RESUME_RUN = True
FORCE_REBUILD = False

RUN_NAME = f"selected_publication_m{M}_r{R_SPAN}_user_spec"
OUTPUT_ROOT = default_output_root(PROJECT_ROOT)

grammar_cfg = SyntheticGrammarConfig(
    vocab_size=M,
    keep_prob_2gram=0.50,
    keep_prob_3gram=0.25,
    exact_support=False,
    support_cache_limit=4096,
)

experiment_cfg = ExperimentConfig(
    sequence_length=40,
    rounds=ROUNDS,
    candidate_count=CANDIDATES_PER_ROUND,
    evaluation_count=EVALUATION_COUNT,
    search_trials=SEARCH_TRIALS,
    prefix_length=2,
    learner_order=3,
    branch_factor=6,
    max_expansions=2500,
)

# Set these to explicit tuples if you want fixed motifs instead of an automatically sampled pattern library.
DESIRABLE_PATTERNS = None
UNDESIRABLE_PATTERNS = None

pattern_sampler_kwargs = {
    "span": R_SPAN,
    "desirable_count": DESIRABLE_COUNT,
    "undesirable_count": UNDESIRABLE_COUNT,
    "pool_sequences": 1500,
    "sequence_length": experiment_cfg.sequence_length,
    "min_count": 2,
    "selection_pool_size": 200,
}

neutral_policy = PublicationPolicy(
    name="publish_all",
    publish_neutral_prob=1.0,
    publish_desirable_prob=1.0,
    publish_undesirable_prob=1.0,
    publish_mixed_prob=1.0,
)

selected_policy = PublicationPolicy(
    name="selected",
    publish_neutral_prob=1.0,
    publish_desirable_prob=1.0,
    publish_undesirable_prob=0.0,
    publish_mixed_prob=0.0,
)

if DESIRABLE_PATTERNS is None or UNDESIRABLE_PATTERNS is None:
    explicit_patterns = None
else:
    explicit_patterns = PatternLibrary(
        span=R_SPAN,
        desirable=tuple(tuple(x) for x in DESIRABLE_PATTERNS),
        undesirable=tuple(tuple(x) for x in UNDESIRABLE_PATTERNS),
    )


## Run or resume the experiment

This uses checkpointed state on disk. If the kernel or notebook stops mid-run, re-running this cell with `RESUME_RUN = True` picks up from the last completed policy/round.


In [ ]:
run = run_publication_experiment(
    version=EXPERIMENT_VERSION,
    run_name=RUN_NAME,
    grammar_config=grammar_cfg,
    experiment_config=experiment_cfg,
    policies=(neutral_policy, selected_policy),
    seed=SEED,
    patterns=explicit_patterns,
    output_root=OUTPUT_ROOT,
    pattern_sampler_kwargs=pattern_sampler_kwargs,
    resume=RESUME_RUN,
    force_rebuild=FORCE_REBUILD,
    progress_bar=True,
)

results = pd.DataFrame(run.table_rows).sort_values(["policy", "round"]).reset_index(drop=True)
patterns_table = pd.DataFrame(
    [{"kind": "desirable", "pattern": format_pattern(pattern)} for pattern in run.result.patterns.desirable]
    + [{"kind": "undesirable", "pattern": format_pattern(pattern)} for pattern in run.result.patterns.undesirable]
)

display(Markdown(f"**Run directory:** `{run.paths.run_dir}`"))
display(Markdown(f"**Checkpoint:** `{run.paths.checkpoint_path}`"))
display(Markdown(f"**Metrics CSV:** `{run.paths.metrics_final_path}`"))
display(Markdown(f"**Summary JSON:** `{run.paths.summary_path}`"))

max_candidate_signal = results[["candidate_desirable_rate", "candidate_undesirable_rate"]].max().max()
if max_candidate_signal < 0.02:
    display(Markdown(
        "**Signal warning:** exact 5-gram hits are sparse for this configuration. "
        "If you want a stronger theorem-2 effect, reduce `R_SPAN`, reduce `M`, or use a larger motif family."
    ))

display(patterns_table)
results


In [ ]:
final_round = (
    results.sort_values(["policy", "round"])
    .groupby("policy", as_index=False)
    .tail(1)
    [[
        "policy",
        "round",
        "cumulative_published",
        "learner_desirable_rate",
        "learner_undesirable_rate",
        "learner_avg_desirable_hits",
        "learner_avg_undesirable_hits",
        "search_success_rate",
        "greedy_success_rate",
    ]]
    .reset_index(drop=True)
)

final_round


## Saved figures

The helper saves figures to disk with the version tag and timestamps baked into the titles, so the notebook is not the only place where the experiment state lives.


In [ ]:
display(Image(filename=str(run.paths.figures_dir / f"{EXPERIMENT_VERSION.lower()}_key_metrics.png")))
display(Image(filename=str(run.paths.figures_dir / f"{EXPERIMENT_VERSION.lower()}_publication_profile.png")))


In [ ]:
display(Markdown("**Sample sequences from the saved run**"))
print(run.paths.samples_path.read_text(encoding="utf-8"))
